# Convert manufacturing CSV routes to XES event logs

Each CSV in `data/manufacturing/` contains process flow definitions (routes).
Each **route** becomes a **trace** (case).
Each **operation** within a route becomes an **event** — ordered by `ROUTEOPERORDER`.

UPT-level detail (unit process targets within each operation) is preserved as event attributes.

In [1]:
import pandas as pd
import pm4py
from pathlib import Path
import json

DATA_DIR = Path("../../data/villach")
OUTPUT_DIR = Path("../../data/villach")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EXPORT_CSV = DATA_DIR / "processflow_export_2.csv"
LABELS_CSV = DATA_DIR / "processflow_variant_labels.csv"

EXPORT_CSV, LABELS_CSV

(WindowsPath('../../data/villach/processflow_export_2.csv'),
 WindowsPath('../../data/villach/processflow_variant_labels.csv'))

## Filter by Processflow Variant Labels
Load `processflow_variant_labels.csv` and keep only routes whose `PROCESSFLOW_LABEL` is one of the target labels.

In [2]:
TARGET_LABELS = {
    "MOWFRA_ERRZZZ_DIZZZZ",
    "MOSFRA_DESFRA_DILABL",
    "MOSFRA_DESFRA_DIZZZZ",
}

labels_df = pd.read_csv(LABELS_CSV, sep=";", dtype=str, skiprows=1).fillna("")
print(f"Loaded {len(labels_df)} label rows")
print(f"Unique labels: {labels_df['PROCESSFLOW_LABEL'].unique().tolist()}")

filtered_labels = labels_df[labels_df["PROCESSFLOW_LABEL"].isin(TARGET_LABELS)]
route_filter = (
    filtered_labels[["FACILITY_NAME", "ROUTE_NAME"]]
    .drop_duplicates()
    .rename(columns={"FACILITY_NAME": "FACILITY", "ROUTE_NAME": "ROUTE"})
)
print(f"\nKept {len(route_filter)} unique (FACILITY, ROUTE) pairs")


Loaded 53 label rows
Unique labels: ['MOWFRA_ERRZZZ_DIZZZZ', 'MOSFRA_DESFRA_DILABL', 'MOSFRA_DESFRA_DIZZZZ', 'MOSFRA_DESFRA_EXLABL', 'MOSFRA_DESFRA_EXLABL_RTCFRA', 'MOSFRA_DESFRA_DILABL_RTCFRA']

Kept 46 unique (FACILITY, ROUTE) pairs


## Converter Function
Reads a CSV, keeps only route-level columns, deduplicates, and exports to XES (ROUTEID = case, ROUTEOPERORDER = order).

In [3]:
def csv_to_xes(csv_path: Path, route_filter: pd.DataFrame | None = None) -> None:
    # auto-detect delimiter (comma default, fallback to semicolon)
    first_line = open(csv_path, encoding="utf-8").readline()
    sep = ";" if ";" in first_line else ","
    df = pd.read_csv(csv_path, sep=sep, dtype=str).fillna("")

    # keep only route-level columns
    keep_cols = ["ROUTERELATION", "FACILITY", "ROUTEID", "ROUTE",
                 "ROUTEDESCRIPTION", "ROUTEOPERORDER", "OPERATION", 'PROCESSAREA']
    df = df[keep_cols]

    # filter by allowed routes if provided
    if route_filter is not None:
        before = len(df)
        df = df.merge(route_filter, on=["FACILITY", "ROUTE"], how="inner")
        print(f"    filtered {before - len(df)} rows outside target labels")

    # drop exact duplicate rows
    before = len(df)
    df = df.drop_duplicates()
    if len(df) < before:
        print(f"    dropped {before - len(df)} duplicate row(s)")

    # when same key but different PROCESSAREA, keep the most frequent one
    key_cols = ["ROUTERELATION", "FACILITY", "ROUTEID", "ROUTE",
                "ROUTEDESCRIPTION", "ROUTEOPERORDER", "OPERATION"]
    before = len(df)
    df["_pa_freq"] = df.groupby(key_cols)["PROCESSAREA"].transform(
        lambda s: s.map(s.value_counts())
    )
    df = (
        df.sort_values("_pa_freq", ascending=False)
          .drop_duplicates(subset=key_cols, keep="first")
          .drop(columns="_pa_freq")
    )
    if len(df) < before:
        print(f"    resolved {before - len(df)} PROCESSAREA conflicts")

    # ensure sortable numeric type for ordering
    df["ROUTEOPERORDER"] = pd.to_numeric(df["ROUTEOPERORDER"], errors="coerce").fillna(0).astype(int)

    # sort by case (ROUTEID) then order
    df = df.sort_values(["ROUTEID", "ROUTEOPERORDER"]).reset_index(drop=True)

    # synthetic timestamps
    step_per_trace = df.groupby("ROUTEID").cumcount()
    df["Timestamp"] = pd.Timestamp("2024-01-01") + pd.to_timedelta(step_per_trace * 2 + 8, unit="h")

    # PM4Py-standard column names
    df["case:concept:name"] = df["ROUTEID"]
    df["concept:name"] = df["OPERATION"]
    df["time:timestamp"] = df["Timestamp"]

    keep = ["case:concept:name", "concept:name", "time:timestamp",
            "ROUTEOPERORDER", "ROUTERELATION", "FACILITY",
            "ROUTE", "ROUTEDESCRIPTION", 'PROCESSAREA']
    events = df[keep].copy()

    # save filtered CSV before XES conversion
    csv_out = OUTPUT_DIR / (csv_path.stem + "_filtered.csv")
    df.to_csv(csv_out, index=False)
    print(f"  {csv_out.name}  ({df['ROUTEID'].nunique()} routes, {len(df)} rows)")

    # build event log
    log = pm4py.convert_to_event_log(events)

    out_path = OUTPUT_DIR / (csv_path.stem + ".xes")
    pm4py.write_xes(log, str(out_path))

    print(f"  {out_path.name}  ({len(log)} traces, {len(events)} events)")
    return log

## Process All CSVs
Iterate over each CSV file and convert it to XES.

In [4]:
print(f"Processing {EXPORT_CSV.name}...")
csv_to_xes(EXPORT_CSV, route_filter=route_filter)

Processing processflow_export_2.csv...
    filtered 370 rows outside target labels
    dropped 1138 duplicate row(s)
    resolved 158 PROCESSAREA conflicts
  processflow_export_2_filtered.csv  (46 routes, 694 rows)


c:\Users\safaya\Documents\GitHub\process-fragment-miner\.venv\Lib\site-packages\pm4py\utils.py:1027: UserWarning: Install the optional requirement `r4pm` to import/export files faster. `rustxes` remains supported as a fallback.
  warnings.warn(


exporting log, completed traces ::   0%|          | 0/46 [00:00<?, ?it/s]

  processflow_export_2.xes  (46 traces, 694 events)


[{'attributes': {'concept:name': '12276866'}, 'events': [{'concept:name': '1017', 'time:timestamp': Timestamp('2024-01-01 08:00:00'), 'ROUTEOPERORDER': 1, 'ROUTERELATION': 'PF-MONV3', 'FACILITY': 'MONV3', 'ROUTE': 'PREC5174', 'ROUTEDESCRIPTION': 'PREC5174', 'PROCESSAREA': 'PRE'}, '..', {'concept:name': '9996', 'time:timestamp': Timestamp('2024-01-02 12:00:00'), 'ROUTEOPERORDER': 15, 'ROUTERELATION': 'PF-MONV3', 'FACILITY': 'MONV3', 'ROUTE': 'PREC5174', 'ROUTEDESCRIPTION': 'PREC5174', 'PROCESSAREA': 'PRE'}]}, '....', {'attributes': {'concept:name': '7286148'}, 'events': [{'concept:name': '1017', 'time:timestamp': Timestamp('2024-01-01 08:00:00'), 'ROUTEOPERORDER': 1, 'ROUTERELATION': 'PF-MONV3', 'FACILITY': 'MONV3', 'ROUTE': 'PREC5039', 'ROUTEDESCRIPTION': 'PREC5032', 'PROCESSAREA': 'PRE'}, '..', {'concept:name': '9996', 'time:timestamp': Timestamp('2024-01-02 12:00:00'), 'ROUTEOPERORDER': 15, 'ROUTERELATION': 'PF-MONV3', 'FACILITY': 'MONV3', 'ROUTE': 'PREC5039', 'ROUTEDESCRIPTION': 'PR

## Verify Output
List all generated XES files in the output directory.

In [5]:
# verify all three files were written
list(OUTPUT_DIR.glob("*.xes"))

[WindowsPath('../../data/villach/processflow_export.xes'),
 WindowsPath('../../data/villach/processflow_export_2.xes')]